In [2]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("Cleaned_Payment_Revenue_Data.csv")

# --------------------------------------------------
# 1. Basic Data Preparation
# --------------------------------------------------

df["Payment_Date"] = pd.to_datetime(
    df["Payment_Date"],
    errors="coerce"
)

df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)

df["Processing_Time_Min"] = pd.to_numeric(
    df["Processing_Time_Min"],
    errors="coerce"
)

# Create date-related columns
df["Year"] = df["Payment_Date"].dt.year
df["Month"] = df["Payment_Date"].dt.month
df["Month_Name"] = df["Payment_Date"].dt.strftime("%b")

# --------------------------------------------------
# 2. Dataset Overview
# --------------------------------------------------

total_records = len(df)
total_columns = len(df.columns)

unique_payments = df["Payment_ID"].nunique()
unique_customers = df["Customer_ID"].nunique()

date_start = df["Payment_Date"].min()
date_end = df["Payment_Date"].max()

overview = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Columns",
        "Unique Payments",
        "Unique Customers",
        "Data Start Date",
        "Data End Date"
    ],
    "Value": [
        total_records,
        total_columns,
        unique_payments,
        unique_customers,
        date_start,
        date_end
    ]
})

# --------------------------------------------------
# 3. Data Quality Analysis
# --------------------------------------------------

missing_values = df.isnull().sum().sum()
duplicate_records = df.duplicated().sum()

quality = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Missing Values",
        "Duplicate Records",
        "Unique Payment IDs",
        "Unique Customer IDs"
    ],
    "Value": [
        len(df),
        missing_values,
        duplicate_records,
        df["Payment_ID"].nunique(),
        df["Customer_ID"].nunique()
    ]
})

# --------------------------------------------------
# 4. Revenue Metric Definitions
# --------------------------------------------------

total_transaction_value = df["Amount"].sum()

successful_df = df[
    df["Payment_Status"] == "Success"
].copy()

failed_df = df[
    df["Payment_Status"] == "Failed"
].copy()

pending_df = df[
    df["Payment_Status"] == "Pending"
].copy()

refunded_df = df[
    df["Payment_Status"] == "Refunded"
].copy()

successful_revenue = successful_df["Amount"].sum()
failed_amount = failed_df["Amount"].sum()
pending_amount = pending_df["Amount"].sum()
refunded_amount = refunded_df["Amount"].sum()

total_transactions = len(df)

successful_transactions = len(successful_df)
failed_transactions = len(failed_df)
pending_transactions = len(pending_df)
refunded_transactions = len(refunded_df)

success_rate = (
    successful_transactions / total_transactions * 100
    if total_transactions > 0 else 0
)

failure_rate = (
    failed_transactions / total_transactions * 100
    if total_transactions > 0 else 0
)

refund_rate = (
    refunded_transactions / total_transactions * 100
    if total_transactions > 0 else 0
)

pending_rate = (
    pending_transactions / total_transactions * 100
    if total_transactions > 0 else 0
)

average_payment = df["Amount"].mean()

average_successful_payment = (
    successful_df["Amount"].mean()
    if len(successful_df) > 0 else 0
)

average_processing_time = (
    df["Processing_Time_Min"].mean()
)

kpis = pd.DataFrame({
    "KPI": [
        "Total Transaction Value",
        "Successful Revenue",
        "Failed Amount",
        "Pending Amount",
        "Refunded Amount",
        "Total Transactions",
        "Successful Transactions",
        "Failed Transactions",
        "Pending Transactions",
        "Refunded Transactions",
        "Success Rate (%)",
        "Failure Rate (%)",
        "Pending Rate (%)",
        "Refund Rate (%)",
        "Average Payment Amount",
        "Average Successful Payment",
        "Average Processing Time (Min)"
    ],
    "Value": [
        round(total_transaction_value, 2),
        round(successful_revenue, 2),
        round(failed_amount, 2),
        round(pending_amount, 2),
        round(refunded_amount, 2),
        total_transactions,
        successful_transactions,
        failed_transactions,
        pending_transactions,
        refunded_transactions,
        round(success_rate, 2),
        round(failure_rate, 2),
        round(pending_rate, 2),
        round(refund_rate, 2),
        round(average_payment, 2),
        round(average_successful_payment, 2),
        round(average_processing_time, 2)
    ]
})

# --------------------------------------------------
# 5. Payment Status Analysis
# --------------------------------------------------

status_analysis = (
    df.groupby("Payment_Status")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean"),
          Average_Processing_Time=("Processing_Time_Min", "mean")
      )
      .reset_index()
)

status_analysis["Transaction_Share"] = (
    status_analysis["Transactions"] /
    status_analysis["Transactions"].sum() * 100
)

status_analysis["Revenue_Share"] = (
    status_analysis["Total_Amount"] /
    status_analysis["Total_Amount"].sum() * 100
)

status_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Transaction_Share",
        "Revenue_Share"
    ]
] = status_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Transaction_Share",
        "Revenue_Share"
    ]
].round(2)

# --------------------------------------------------
# 6. Gateway Analysis
# --------------------------------------------------

gateway_analysis = (
    df.groupby("Gateway")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean"),
          Average_Processing_Time=("Processing_Time_Min", "mean")
      )
      .reset_index()
)

gateway_success = (
    df.assign(
        Success=df["Payment_Status"].eq("Success").astype(int)
    )
    .groupby("Gateway")["Success"]
    .mean()
    .mul(100)
    .reset_index(name="Success_Rate")
)

gateway_analysis = gateway_analysis.merge(
    gateway_success,
    on="Gateway",
    how="left"
)

gateway_analysis["Revenue_Share"] = (
    gateway_analysis["Total_Amount"] /
    gateway_analysis["Total_Amount"].sum() * 100
)

gateway_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Success_Rate",
        "Revenue_Share"
    ]
] = gateway_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Success_Rate",
        "Revenue_Share"
    ]
].round(2)

# --------------------------------------------------
# 7. Payment Method Analysis
# --------------------------------------------------

method_analysis = (
    df.groupby("Payment_Method")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean"),
          Average_Processing_Time=("Processing_Time_Min", "mean")
      )
      .reset_index()
)

method_success = (
    df.assign(
        Success=df["Payment_Status"].eq("Success").astype(int)
    )
    .groupby("Payment_Method")["Success"]
    .mean()
    .mul(100)
    .reset_index(name="Success_Rate")
)

method_analysis = method_analysis.merge(
    method_success,
    on="Payment_Method",
    how="left"
)

method_analysis["Revenue_Share"] = (
    method_analysis["Total_Amount"] /
    method_analysis["Total_Amount"].sum() * 100
)

method_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Success_Rate",
        "Revenue_Share"
    ]
] = method_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Average_Processing_Time",
        "Success_Rate",
        "Revenue_Share"
    ]
].round(2)

# --------------------------------------------------
# 8. Region Analysis
# --------------------------------------------------

region_analysis = (
    df.groupby("Region")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean")
      )
      .reset_index()
)

region_success = (
    df.assign(
        Success=df["Payment_Status"].eq("Success").astype(int)
    )
    .groupby("Region")["Success"]
    .mean()
    .mul(100)
    .reset_index(name="Success_Rate")
)

region_analysis = region_analysis.merge(
    region_success,
    on="Region",
    how="left"
)

region_analysis["Revenue_Share"] = (
    region_analysis["Total_Amount"] /
    region_analysis["Total_Amount"].sum() * 100
)

region_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Success_Rate",
        "Revenue_Share"
    ]
] = region_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Success_Rate",
        "Revenue_Share"
    ]
].round(2)

# --------------------------------------------------
# 9. Customer Type Analysis
# --------------------------------------------------

customer_type_analysis = (
    df.groupby("Customer_Type")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean")
      )
      .reset_index()
)

customer_success = (
    df.assign(
        Success=df["Payment_Status"].eq("Success").astype(int)
    )
    .groupby("Customer_Type")["Success"]
    .mean()
    .mul(100)
    .reset_index(name="Success_Rate")
)

customer_type_analysis = customer_type_analysis.merge(
    customer_success,
    on="Customer_Type",
    how="left"
)

customer_type_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Success_Rate"
    ]
] = customer_type_analysis[
    [
        "Total_Amount",
        "Average_Amount",
        "Success_Rate"
    ]
].round(2)

# --------------------------------------------------
# 10. Monthly Revenue Analysis
# --------------------------------------------------

monthly_analysis = (
    df.groupby(
        ["Year", "Month", "Month_Name"]
    )
    .agg(
        Transactions=("Payment_ID", "count"),
        Total_Amount=("Amount", "sum"),
        Average_Amount=("Amount", "mean")
    )
    .reset_index()
    .sort_values(["Year", "Month"])
)

monthly_analysis[
    [
        "Total_Amount",
        "Average_Amount"
    ]
] = monthly_analysis[
    [
        "Total_Amount",
        "Average_Amount"
    ]
].round(2)

# --------------------------------------------------
# 11. Monthly Successful Revenue
# --------------------------------------------------

monthly_successful_revenue = (
    successful_df.groupby(
        ["Year", "Month", "Month_Name"]
    )
    .agg(
        Successful_Transactions=("Payment_ID", "count"),
        Successful_Revenue=("Amount", "sum"),
        Average_Successful_Payment=("Amount", "mean")
    )
    .reset_index()
    .sort_values(["Year", "Month"])
)

monthly_successful_revenue[
    [
        "Successful_Revenue",
        "Average_Successful_Payment"
    ]
] = monthly_successful_revenue[
    [
        "Successful_Revenue",
        "Average_Successful_Payment"
    ]
].round(2)

# --------------------------------------------------
# 12. Monthly Success Rate
# --------------------------------------------------

monthly_success_rate = (
    df.assign(
        Success=df["Payment_Status"].eq("Success").astype(int)
    )
    .groupby(["Year", "Month", "Month_Name"])
    .agg(
        Transactions=("Payment_ID", "count"),
        Successful_Transactions=("Success", "sum")
    )
    .reset_index()
    .sort_values(["Year", "Month"])
)

monthly_success_rate["Success_Rate"] = (
    monthly_success_rate["Successful_Transactions"] /
    monthly_success_rate["Transactions"] * 100
).round(2)

# --------------------------------------------------
# 13. Processing Time Analysis
# --------------------------------------------------

processing_analysis = (
    df.groupby("Payment_Status")
      .agg(
          Average_Processing_Time=("Processing_Time_Min", "mean"),
          Minimum_Processing_Time=("Processing_Time_Min", "min"),
          Maximum_Processing_Time=("Processing_Time_Min", "max")
      )
      .reset_index()
)

processing_analysis[
    [
        "Average_Processing_Time",
        "Minimum_Processing_Time",
        "Maximum_Processing_Time"
    ]
] = processing_analysis[
    [
        "Average_Processing_Time",
        "Minimum_Processing_Time",
        "Maximum_Processing_Time"
    ]
].round(2)

# --------------------------------------------------
# 14. Top Customers
# --------------------------------------------------

top_customers = (
    df.groupby("Customer_ID")
      .agg(
          Transactions=("Payment_ID", "count"),
          Total_Amount=("Amount", "sum"),
          Average_Amount=("Amount", "mean")
      )
      .reset_index()
      .sort_values(
          "Total_Amount",
          ascending=False
      )
      .head(10)
)

top_customers[
    [
        "Total_Amount",
        "Average_Amount"
    ]
] = top_customers[
    [
        "Total_Amount",
        "Average_Amount"
    ]
].round(2)

# --------------------------------------------------
# 15. Gateway Status Analysis
# --------------------------------------------------

gateway_status = (
    df.groupby(
        ["Gateway", "Payment_Status"]
    )
    .agg(
        Transactions=("Payment_ID", "count"),
        Total_Amount=("Amount", "sum")
    )
    .reset_index()
)

# --------------------------------------------------
# 16. Revenue Metric Dictionary
# --------------------------------------------------

metric_dictionary = pd.DataFrame({
    "Metric": [
        "Total Transaction Value",
        "Successful Revenue",
        "Success Rate",
        "Failure Rate",
        "Pending Rate",
        "Refund Rate",
        "Average Payment Amount",
        "Average Processing Time",
        "Gateway Success Rate",
        "Payment Method Success Rate"
    ],
    "Definition": [
        "Total amount across all payment transactions.",
        "Total amount from successful payment transactions.",
        "Successful transactions divided by total transactions.",
        "Failed transactions divided by total transactions.",
        "Pending transactions divided by total transactions.",
        "Refunded transactions divided by total transactions.",
        "Average payment amount across transactions.",
        "Average time taken to process a payment.",
        "Successful gateway transactions divided by gateway transactions.",
        "Successful payment-method transactions divided by payment-method transactions."
    ],
    "Source_Field": [
        "Amount",
        "Amount + Payment_Status",
        "Payment_Status",
        "Payment_Status",
        "Payment_Status",
        "Payment_Status",
        "Amount",
        "Processing_Time_Min",
        "Gateway + Payment_Status",
        "Payment_Method + Payment_Status"
    ]
})

# --------------------------------------------------
# 17. Business Insights
# --------------------------------------------------

best_gateway_row = gateway_analysis.loc[
    gateway_analysis["Success_Rate"].idxmax()
]

best_method_row = method_analysis.loc[
    method_analysis["Success_Rate"].idxmax()
]

highest_region_row = region_analysis.loc[
    region_analysis["Total_Amount"].idxmax()
]

highest_month_row = monthly_analysis.loc[
    monthly_analysis["Total_Amount"].idxmax()
]

slowest_gateway_row = gateway_analysis.loc[
    gateway_analysis["Average_Processing_Time"].idxmax()
]

highest_customer_row = top_customers.iloc[0]

insights = pd.DataFrame({
    "Business Insight": [
        "Total Transaction Value",
        "Successful Revenue",
        "Overall Success Rate",
        "Overall Failure Rate",
        "Best Performing Gateway",
        "Best Gateway Success Rate",
        "Best Performing Payment Method",
        "Best Payment Method Success Rate",
        "Highest Revenue Region",
        "Highest Revenue Region Amount",
        "Highest Revenue Month",
        "Highest Monthly Revenue",
        "Slowest Gateway by Processing Time",
        "Highest Value Customer"
    ],
    "Value": [
        round(total_transaction_value, 2),
        round(successful_revenue, 2),
        round(success_rate, 2),
        round(failure_rate, 2),
        best_gateway_row["Gateway"],
        round(best_gateway_row["Success_Rate"], 2),
        best_method_row["Payment_Method"],
        round(best_method_row["Success_Rate"], 2),
        highest_region_row["Region"],
        round(highest_region_row["Total_Amount"], 2),
        highest_month_row["Month_Name"],
        round(highest_month_row["Total_Amount"], 2),
        slowest_gateway_row["Gateway"],
        highest_customer_row["Customer_ID"]
    ]
})

# --------------------------------------------------
# 18. Final Analysis Output
# --------------------------------------------------

output_file = "Python Analysis Result.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    overview.to_excel(
        writer,
        sheet_name="Overview",
        index=False
    )

    quality.to_excel(
        writer,
        sheet_name="Data Quality",
        index=False
    )

    kpis.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    status_analysis.to_excel(
        writer,
        sheet_name="Payment Status",
        index=False
    )

    gateway_analysis.to_excel(
        writer,
        sheet_name="Gateway Analysis",
        index=False
    )

    method_analysis.to_excel(
        writer,
        sheet_name="Payment Method",
        index=False
    )

    region_analysis.to_excel(
        writer,
        sheet_name="Region Analysis",
        index=False
    )

    customer_type_analysis.to_excel(
        writer,
        sheet_name="Customer Type",
        index=False
    )

    monthly_analysis.to_excel(
        writer,
        sheet_name="Monthly Revenue",
        index=False
    )

    monthly_successful_revenue.to_excel(
        writer,
        sheet_name="Successful Revenue",
        index=False
    )

    monthly_success_rate.to_excel(
        writer,
        sheet_name="Monthly Success Rate",
        index=False
    )

    processing_analysis.to_excel(
        writer,
        sheet_name="Processing Time",
        index=False
    )

    top_customers.to_excel(
        writer,
        sheet_name="Top Customers",
        index=False
    )

    gateway_status.to_excel(
        writer,
        sheet_name="Gateway Status",
        index=False
    )

    metric_dictionary.to_excel(
        writer,
        sheet_name="Metric Dictionary",
        index=False
    )

    insights.to_excel(
        writer,
        sheet_name="Business Insights",
        index=False
    )

print("Analysis completed successfully.")
print()
print("Final output file:")
print("Python Analysis Result.xlsx")

Analysis completed successfully.

Final output file:
Python Analysis Result.xlsx
